In [47]:
import pandas as pd


In [48]:
df = pd.read_csv("/Users/kynesantos/Desktop/Grad School/Trade Networks/trade_1988_2021.csv")

In [49]:
df.head()

,ReporterISO3,ReporterName,PartnerISO3,PartnerName,Year,TradeFlowName,TradeValue in 1000 USD
0,AFG,Afghanistan,SWE,Sweden,2017,Export,86.752
1,AFG,Afghanistan,JOR,Jordan,2018,Export,2796.481
2,AFG,Afghanistan,JOR,Jordan,2017,Export,3100.187
3,AFG,Afghanistan,ITA,Italy,2018,Export,279.918
4,AFG,Afghanistan,ITA,Italy,2017,Export,416.642


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 634509 entries, 0 to 634508
Data columns (total 7 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   ReporterISO3            634509 non-null  object 
 1   ReporterName            634509 non-null  object 
 2   PartnerISO3             634509 non-null  object 
 3   PartnerName             634509 non-null  object 
 4   Year                    634509 non-null  int64  
 5   TradeFlowName           634509 non-null  object 
 6   TradeValue in 1000 USD  634441 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 33.9+ MB


In [51]:
merged = df.merge(
    df,
    left_on=["ReporterISO3", "PartnerISO3", "Year"],
    right_on=["PartnerISO3", "ReporterISO3", "Year"],
    suffixes=("_AtoB", "_BtoA"),
    how="left"
)


In [52]:
merged.head()

,ReporterISO3_AtoB,ReporterName_AtoB,PartnerISO3_AtoB,PartnerName_AtoB,Year,TradeFlowName_AtoB,TradeValue in 1000 USD_AtoB,ReporterISO3_BtoA,ReporterName_BtoA,PartnerISO3_BtoA,PartnerName_BtoA,TradeFlowName_BtoA,TradeValue in 1000 USD_BtoA
0,AFG,Afghanistan,SWE,Sweden,2017,Export,86.752,SWE,Sweden,AFG,Afghanistan,Export,13958.154
1,AFG,Afghanistan,JOR,Jordan,2018,Export,2796.481,JOR,Jordan,AFG,Afghanistan,Export,974.457
2,AFG,Afghanistan,JOR,Jordan,2017,Export,3100.187,JOR,Jordan,AFG,Afghanistan,Export,1533.252
3,AFG,Afghanistan,ITA,Italy,2018,Export,279.918,ITA,Italy,AFG,Afghanistan,Export,13227.168
4,AFG,Afghanistan,ITA,Italy,2017,Export,416.642,ITA,Italy,AFG,Afghanistan,Export,18783.400


In [53]:
merged2 = merged.drop(columns=['TradeFlowName_AtoB', 
'ReporterISO3_BtoA', 'ReporterName_BtoA', 'PartnerISO3_BtoA', 'PartnerName_BtoA','TradeFlowName_BtoA'])
merged2.head()

,ReporterISO3_AtoB,ReporterName_AtoB,PartnerISO3_AtoB,PartnerName_AtoB,Year,TradeValue in 1000 USD_AtoB,TradeValue in 1000 USD_BtoA
0,AFG,Afghanistan,SWE,Sweden,2017,86.752,13958.154
1,AFG,Afghanistan,JOR,Jordan,2018,2796.481,974.457
2,AFG,Afghanistan,JOR,Jordan,2017,3100.187,1533.252
3,AFG,Afghanistan,ITA,Italy,2018,279.918,13227.168
4,AFG,Afghanistan,ITA,Italy,2017,416.642,18783.400


In [54]:
merged2 = merged2.rename(columns={'ReporterISO3_AtoB': 'Country A','ReporterName_AtoB': 'Country A Name', 'PartnerISO3_AtoB': 'Country B', 'PartnerName_AtoB': 'Country B Name','TradeValue in 1000 USD_AtoB':'Exports', 'TradeValue in 1000 USD_BtoA':'Imports'})
merged2.head()

,Country A,Country A Name,Country B,Country B Name,Year,Exports,Imports
0,AFG,Afghanistan,SWE,Sweden,2017,86.752,13958.154
1,AFG,Afghanistan,JOR,Jordan,2018,2796.481,974.457
2,AFG,Afghanistan,JOR,Jordan,2017,3100.187,1533.252
3,AFG,Afghanistan,ITA,Italy,2018,279.918,13227.168
4,AFG,Afghanistan,ITA,Italy,2017,416.642,18783.400


In [55]:
merged2['Surplus'] = merged2['Exports'] - merged2['Imports']
merged2.head()

,Country A,Country A Name,Country B,Country B Name,Year,Exports,Imports,Surplus
0,AFG,Afghanistan,SWE,Sweden,2017,86.752,13958.154,-13871.402
1,AFG,Afghanistan,JOR,Jordan,2018,2796.481,974.457,1822.024
2,AFG,Afghanistan,JOR,Jordan,2017,3100.187,1533.252,1566.935
3,AFG,Afghanistan,ITA,Italy,2018,279.918,13227.168,-12947.250
4,AFG,Afghanistan,ITA,Italy,2017,416.642,18783.400,-18366.758


In [56]:
surplus = merged2[merged2['Surplus'] > 0]
surplus.head()
surplus.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200982 entries, 1 to 634503
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country A       200982 non-null  object 
 1   Country A Name  200982 non-null  object 
 2   Country B       200982 non-null  object 
 3   Country B Name  200982 non-null  object 
 4   Year            200982 non-null  int64  
 5   Exports         200982 non-null  float64
 6   Imports         200982 non-null  float64
 7   Surplus         200982 non-null  float64
dtypes: float64(3), int64(1), object(4)
memory usage: 13.8+ MB


In [57]:
edges = surplus.rename(columns={
    "Country A": "Source",
    "Country B": "Target",
    "Surplus": "Weight"
})

In [58]:
edges.head()

,Source,Country A Name,Target,Country B Name,Year,Exports,Imports,Weight
1,AFG,Afghanistan,JOR,Jordan,2018,2796.481,974.457,1822.024
2,AFG,Afghanistan,JOR,Jordan,2017,3100.187,1533.252,1566.935
22,AFG,Afghanistan,FIN,Finland,2018,1228.017,499.134,728.883
23,AFG,Afghanistan,FIN,Finland,2017,3784.262,615.458,3168.804
35,AFG,Afghanistan,SAU,Saudi Arabia,2018,6604.192,4342.950,2261.242


In [59]:
nodes = edges[["Source"]].drop_duplicates()

In [60]:
nodes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 204 entries, 1 to 630967
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Source  204 non-null    object
dtypes: object(1)
memory usage: 3.2+ KB


In [61]:
nodes.head()

,Source
1,AFG
471,ALB
2533,DZA
5156,AND
6967,AGO


In [62]:
edges.to_csv("edges.csv", index=False)
nodes.to_csv("nodes.csv", index=False)